# Phase 2 — Learned detector (Colab, GPU runtime)

Trains a frozen-DINOv2 + attention-pooling detector, evaluates cross-generator
transfer, and wires the result into the fusion pipeline. All logic lives in
repo modules (`training/`, `ai_image_id/detector/`); this notebook orchestrates.

**Runtime → Change runtime type → GPU (T4).**

Storage policy: immutable sources (GenImage zips) live on the shared Drive;
small versioned artifacts (checkpoints, calibration) go to your Drive under
`ai_image_id/runs/<commit>/`; large derived caches (embeddings) go to
`ai_image_id/emb/` on Drive so they survive session resets. Extracted images
and prepared splits are ephemeral — session disk only, deleted after embedding.
Absolute paths everywhere (`/content/...`).

In [9]:
# ── Setup — fresh-kernel-proof ──────────────────────────────────
!apt-get -qq install -y libimage-exiftool-perl p7zip-full > /dev/null
!git clone -q https://github.com/Waranika/AI-image-Checkers.git 2>/dev/null || echo "already cloned"
%cd /content/AI-image-Checkers
%pip install -q -e .
import sys; sys.path.insert(0, "/content/AI-image-Checkers")

import torch
print(torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "→ Runtime > GPU")

/content/AI-image-Checkers
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ai-image-id (pyproject.toml) ... done
2.11.0+cu128 | cuda: True | Tesla T4


## (Optional) Smoke test — full torch chain on synthetic data, ~2 min

No downloads, no Drive. Exercises `prepare_split → precompute → train →
calibrate` end-to-end on 60 synthetic images/class. Numbers are meaningless;
execution is everything.

In [10]:
import numpy as np
from pathlib import Path
from PIL import Image

rng = np.random.default_rng(0)
for cls in ["real", "fake"]:
    d = Path(f"/content/smoke/src_{cls}"); d.mkdir(parents=True, exist_ok=True)
    for i in range(60):
        y, x = np.mgrid[0:384, 0:384]
        f1, f2 = rng.uniform(40, 140, 2)
        img = np.stack([120+60*np.sin(x/f1), 100+50*np.cos(y/f2), 90+40*np.sin((x+y)/97)], axis=-1)
        noise = rng.normal(0, 6 if cls == "real" else 2, img.shape)
        Image.fromarray(np.clip(img+noise, 0, 255).astype(np.uint8)).save(d/f"{i}.png")

from training.prepare_data import prepare_split
from training.embed import precompute
from training.train_head import train
from training.calibrate_eval import calibrate

prepare_split("/content/smoke/src_real", "/content/smoke/src_fake", "/content/smoke/train", n_per_class=48, seed=0)
prepare_split("/content/smoke/src_real", "/content/smoke/src_fake", "/content/smoke/val",   n_per_class=12, seed=1)
precompute("/content/smoke/train/manifest.csv", "/content/smoke/train", "/content/smoke/emb_train", n_aug=1, device="cuda")
precompute("/content/smoke/val/manifest.csv",   "/content/smoke/val",   "/content/smoke/emb_val",   n_aug=0, device="cuda")
ckpt = train("/content/smoke/emb_train", "/content/smoke/emb_val", out="/content/smoke/head.pt", epochs=3)
print(calibrate("/content/smoke/head.val_logits.npz", "/content/smoke/head.calibration.json"))

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 6 shards to /content/smoke/emb_train


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 1 shards to /content/smoke/emb_val
train: 192 samples (0.0 GB fp16), val: 24 samples, dim=384, device=cpu
epoch 0: train_loss=0.7030 val_acc=0.7500
epoch 1: train_loss=0.6223 val_acc=0.7500
epoch 2: train_loss=0.5533 val_acc=0.8750
{'temperature': 0.1128, 'ece_before': 0.1966, 'ece_after': 0.1711, 'nll_before': 0.5289, 'nll_after': 0.3102}


## Step 1 — Data: GenImage SDv1.4, val-split slice

The full SDv1.4 subset is a **30-volume split zip (~90 GB)** — exceeds
session disk. Strategy: extract only `val/*` (6K/class, ~3.6 GB) directly
off the Drive mount. The full `train/` run is a later session.

One-time prerequisite: Drive shortcut to the shared GenImage folder →
`stable_diffusion_v_1_4`.

In [11]:
from google.colab import drive
drive.mount('/content/drive')

SHARED = "/content/drive/MyDrive/stable_diffusion_v_1_4"

!rm -rf /content/zips /content/genimage /content/data
!df -h /content | tail -1
!ls "$SHARED"/imagenet_ai_0419_sdv4.* | wc -l  # expect 30

!7z x -y -o/content/genimage "$SHARED/imagenet_ai_0419_sdv4.zip" \
      "imagenet_ai_0419_sdv4/val/*" | tail -4

!find /content/genimage -maxdepth 4 -type d
from pathlib import Path
root = Path("/content/genimage/imagenet_ai_0419_sdv4")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
overlay         113G   56G   57G  50% /
30
Folders: 2
Files: 12000
Size:       3557533313
Compressed: 96413397770
/content/genimage
/content/genimage/imagenet_ai_0419_sdv4
/content/genimage/imagenet_ai_0419_sdv4/val
/content/genimage/imagenet_ai_0419_sdv4/val/nature
/content/genimage/imagenet_ai_0419_sdv4/val/ai


## Step 1b — De-confounded, disjoint splits

Single prep of the whole val pool, then disjoint slices of the shuffled
manifest — overlap is structurally impossible.

In [12]:
from training.prepare_data import prepare_split
import csv, shutil
from pathlib import Path
from collections import defaultdict

prepare_split(root/"val/nature", root/"val/ai", "/content/data/all",
              n_per_class=6_000, seed=0)

src = Path("/content/data/all")
rows = list(csv.DictReader((src/"manifest.csv").open()))
by_label = defaultdict(list)
for r in rows:
    by_label[r["label"]].append(r)

for split, sl in (("train", slice(0, 5000)), ("val", slice(5000, 6000))):
    out = Path(f"/content/data/{split}")
    out_rows = []
    for label, rs in by_label.items():
        (out/label).mkdir(parents=True, exist_ok=True)
        for r in rs[sl]:
            shutil.move(str(src/label/r["file"]), out/label/r["file"])
            out_rows.append(r)
    with (out/"manifest.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["file","label","src","jpeg_q","short_side"])
        w.writeheader(); w.writerows(out_rows)
shutil.rmtree(src)

train_manifest = "/content/data/train/manifest.csv"
val_manifest   = "/content/data/val/manifest.csv"
train_srcs = {r["src"] for r in csv.DictReader(open(train_manifest))}
val_srcs   = {r["src"] for r in csv.DictReader(open(val_manifest))}
overlap = len(train_srcs & val_srcs)
print(f"train/val source overlap: {overlap} files", "⚠️ LEAKAGE" if overlap else "✓ clean")

train/val source overlap: 0 files ✓ clean


## Step 2 — Precompute frozen embeddings (SDv1.4)

Val first (2K images, n_aug=0, minutes) as the real-data smoke test.
Then train (10K × 3 variants = 30K forward passes).

In [13]:
from training.embed import precompute

precompute("/content/data/val/manifest.csv",   "/content/data/val",   "/content/emb/val",
           n_aug=0, device="cuda")
precompute("/content/data/train/manifest.csv", "/content/data/train", "/content/emb/train",
           n_aug=2, device="cuda")

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 63 shards to /content/emb/val


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 938 shards to /content/emb/train


PosixPath('/content/emb/train')

## Step 3 — Train the head + calibrate

In [14]:
from training.train_head import train
from training.calibrate_eval import calibrate

ckpt = train("/content/emb/train", "/content/emb/val", out="/content/head.pt",
             epochs=5, device="cuda")
report = calibrate("/content/head.val_logits.npz", "/content/head.calibration.json")
print(report)

train: 30000 samples (5.9 GB fp16), val: 2000 samples, dim=384, device=cuda
epoch 0: train_loss=0.2812 val_acc=0.9070
epoch 1: train_loss=0.1392 val_acc=0.9130
epoch 2: train_loss=0.0850 val_acc=0.9195
epoch 3: train_loss=0.0513 val_acc=0.9170
epoch 4: train_loss=0.0285 val_acc=0.9225
{'temperature': 2.6328, 'ece_before': 0.0573, 'ece_after': 0.0132, 'nll_before': 0.3292, 'nll_after': 0.21}


## Step 4 — Persist checkpoint to Drive

In [15]:
import json, shutil, subprocess
from pathlib import Path

commit = subprocess.run(["git", "-C", "/content/AI-image-Checkers", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
RUN = Path(f"/content/drive/MyDrive/ai_image_id/runs/{commit}")
RUN.mkdir(parents=True, exist_ok=True)

for f in ["/content/head.pt", "/content/head.calibration.json",
          "/content/data/train/manifest.csv", "/content/data/val/manifest.csv"]:
    shutil.copy(f, RUN)

(RUN/"provenance.json").write_text(json.dumps({
    "commit": commit, "backbone": "dinov2_vits14",
    "data": "GenImage sdv14 val-split slice", "n_train_per_class": 5000,
    "n_val_per_class": 1000, "n_aug": 2, "seeds": {"prep": 0},
}, indent=2))
print("saved to", RUN)

saved to /content/drive/MyDrive/ai_image_id/runs/8cf8026


## Step 5 — Use the detector in the pipeline

In [16]:
import os, random
from ai_image_id.main import analyze_image

random.seed(0)
fake_dir, real_dir = "/content/data/val/fake", "/content/data/val/real"
samples = [os.path.join(fake_dir, random.choice(os.listdir(fake_dir))),
           os.path.join(real_dir, random.choice(os.listdir(real_dir)))]

for path in samples:
    r = analyze_image(path, detector_ckpt="/content/head.pt")
    d = r.evidence.detector
    print(f"\n{path.split('/')[-2]}/{path.split('/')[-1]}")
    print(f"  verdict: {r.ai_verdict.value} ({r.confidence})")
    print(f"  detector: p={d.p_calibrated}  drift={d.robustness_drift}")
    print(f"  notes: {r.notes}")

preprocessor_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 44.8MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main



fake/fake_005067.jpg
  verdict: inconclusive (0.5)
  detector: p=0.9106  drift=0.0154
  notes: ['no AI-indicative signal found', 'absence of watermarks/metadata is non-evidence, not proof of human origin']

real/real_005481.jpg
  verdict: unlikely (0.6)
  detector: p=0.008  drift=0.0048
  notes: ['learned detector p=0.01 (low) — weak evidence of camera origin; PRNU/reverse-search needed to strengthen']


## Step 6 — Cross-generator evaluation

Extract other generators' `val/*` slices, prepare, embed, and persist
to Drive so future sessions skip the extraction entirely.

**First time:** run the extraction cell (15–30 min per generator over FUSE).
**Returning session:** run only the restore cell (seconds).

### Option A — Restore from Drive (returning session)

In [17]:
import shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

GENS = ["Midjourney", "ADM", "BigGAN", "glide"]
EMB_DRIVE = Path("/content/drive/MyDrive/ai_image_id/emb")

restored, missing = [], []
for gen in GENS:
    src = EMB_DRIVE / gen
    dst = Path(f"/content/eval/{gen}/emb")
    if dst.exists():
        print(f"{gen}: already on session disk"); restored.append(gen); continue
    if not src.exists() or not list(src.glob("shard_*.npz")):
        print(f"{gen}: not on Drive yet — run the extraction cell"); missing.append(gen); continue
    dst.mkdir(parents=True, exist_ok=True)
    for shard in src.glob("shard_*.npz"):
        shutil.copy(shard, dst / shard.name)
    n = len(list(dst.glob("*.npz")))
    print(f"{gen}: restored {n} shards from Drive")
    restored.append(gen)

if missing:
    print(f"\n⚠ {len(missing)} generators need extraction: {missing}")
else:
    print(f"\n✓ all {len(GENS)} generators ready on disk")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Midjourney: not on Drive yet — run the extraction cell
ADM: not on Drive yet — run the extraction cell
BigGAN: not on Drive yet — run the extraction cell
glide: not on Drive yet — run the extraction cell

⚠ 4 generators need extraction: ['Midjourney', 'ADM', 'BigGAN', 'glide']


### Option B — Extract + embed + persist to Drive (first time only)

Only run this for generators NOT already on Drive. Takes 15–30 min per
generator (FUSE extraction is the bottleneck, not GPU). Each generator is
a self-contained transaction: extract → prepare → embed → save to Drive →
delete raw images.

In [18]:
import shutil, subprocess
from pathlib import Path
from training.prepare_data import prepare_split
from training.embed import precompute
from google.colab import drive
drive.mount('/content/drive')

GENS = ["Midjourney", "ADM", "BigGAN", "glide"]
GENIMAGE = "/content/drive/MyDrive/genimage_shared"
EMB_DRIVE = Path("/content/drive/MyDrive/ai_image_id/emb")

for gen in GENS:
    dst_check = Path(f"/content/eval/{gen}/emb")
    drive_check = EMB_DRIVE / gen
    if dst_check.exists() or (drive_check.exists() and list(drive_check.glob("shard_*.npz"))):
        print(f"{gen}: embeddings already exist, skipping"); continue

    src = Path(f"{GENIMAGE}/{gen}")
    masters = sorted(src.glob("*.zip"))
    if not masters:
        raise FileNotFoundError(f"{gen}: no .zip master in {src}")
    master = masters[0]
    workdir = Path(f"/content/eval/{gen}")

    print(f"═══ {gen}: extracting val slice ═══")
    !7z x -y -o/content/eval/{gen}/raw "{master}" "*/val/*" | tail -3

    val = next(p for p in (workdir/"raw").rglob("val") if (p/"ai").is_dir())
    print("val dir:", val)

    prepare_split(val/"nature", val/"ai", workdir/"data", n_per_class=1_000, seed=42)
    precompute(workdir/"data/manifest.csv", workdir/"data", workdir/"emb",
               n_aug=0, device="cuda")

    # Persist to Drive before cleaning up
    drive_dst = EMB_DRIVE / gen
    drive_dst.mkdir(parents=True, exist_ok=True)
    for shard in (workdir/"emb").glob("shard_*.npz"):
        shutil.copy(shard, drive_dst / shard.name)
    print(f"{gen}: {len(list(drive_dst.glob('*.npz')))} shards saved to Drive")

    # Clean up raw images + prepared data (keep only embeddings on session disk)
    shutil.rmtree(workdir/"raw", ignore_errors=True)
    shutil.rmtree(workdir/"data", ignore_errors=True)
    print(f"{gen}: done, raw data cleaned\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
═══ Midjourney: extracting val slice ═══
Files: 12000
Size:       8328212761
Compressed: 227469216710
val dir: /content/eval/Midjourney/raw/imagenet_midjourney/val


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 63 shards to /content/eval/Midjourney/emb
Midjourney: 63 shards saved to Drive
Midjourney: done, raw data cleaned

═══ ADM: extracting val slice ═══
Files: 12000
Size:       1520044631
Compressed: 38839383692
val dir: /content/eval/ADM/raw/imagenet_ai_0508_adm/val


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 63 shards to /content/eval/ADM/emb
ADM: 63 shards saved to Drive
ADM: done, raw data cleaned

═══ BigGAN: extracting val slice ═══
Files: 12000
Size:       982096774
Compressed: 24235936500
val dir: /content/eval/BigGAN/raw/imagenet_ai_0419_biggan/val


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 63 shards to /content/eval/BigGAN/emb
BigGAN: 63 shards saved to Drive
BigGAN: done, raw data cleaned

═══ glide: extracting val slice ═══
Files: 12000
Size:       1256157232
Compressed: 32263862577
val dir: /content/eval/glide/raw/imagenet_glide/val


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 63 shards to /content/eval/glide/emb
glide: 63 shards saved to Drive
glide: done, raw data cleaned



### Cross-generator table

Score each generator's val embeddings with the trained head. The reference
points: in-distribution ~0.92, GenImage's ResNet-50 baseline ~52–55% on
unseen generators.

In [19]:
import json, numpy as np, torch
from training.train_head import build_head, _load_split, _batched_logits
from training.calibrate_eval import cross_generator_table, _sigmoid

state = torch.load("/content/head.pt", map_location="cpu")
head = build_head(state["dim"]); head.load_state_dict(state["state_dict"])
head = head.to("cuda").eval()
T = json.load(open("/content/head.calibration.json"))["temperature"]

GENS = ["Midjourney", "ADM", "BigGAN", "glide"]

def score(emb_dir):
    p, c, y = _load_split(emb_dir)
    logits = _batched_logits(head, p, c, "cuda").numpy().astype(np.float64)
    return _sigmoid(logits / T), y.astype(np.float64)

results = {"sdv14 (in-dist)": score("/content/emb/val")}
for gen in GENS:
    results[gen] = score(f"/content/eval/{gen}/emb")

print(cross_generator_table(results))

| generator | AUROC | bal. acc |
|---|---|---|
| ADM | 0.345 | 0.453 |
| BigGAN | 0.560 | 0.524 |
| Midjourney | 0.832 | 0.700 |
| glide | 0.776 | 0.649 |
| sdv14 (in-dist) | 0.973 | 0.923 |
| **mean** | **0.697** | **0.650** |


## Step 7 — Mixed-generator training (the ablation)

Pool embeddings from architecturally diverse generators into one training
set. The before/after cross-generator table — SDv1.4-only vs. mixed — is
the headline finding: does multi-generator training repair the ADM
inversion and improve transfer across the board?

Uses val-slice embeddings with n_aug=0 (already computed). The existing
`_load_split` handles the combined ~8 GB in RAM.

In [20]:
import shutil
from pathlib import Path

POOL_TRAIN = Path("/content/emb_mixed/train")
POOL_VAL = Path("/content/emb_mixed/val")
shutil.rmtree(POOL_TRAIN, ignore_errors=True)
shutil.rmtree(POOL_VAL, ignore_errors=True)
POOL_TRAIN.mkdir(parents=True, exist_ok=True)
POOL_VAL.mkdir(parents=True, exist_ok=True)

# Pool: SDv1.4 train embeddings + cross-gen eval embeddings (all n_aug=0)
# SDv1.4 uses its own train split; others use their val embeddings for training
# since we only extracted val slices.
for gen in ["Midjourney", "ADM", "BigGAN", "glide"]:
    src = Path(f"/content/eval/{gen}/emb")
    for shard in sorted(src.glob("shard_*.npz")):
        dst = POOL_TRAIN / f"shard_{gen.lower()}_{shard.name}"
        shutil.copy(shard, dst)
    print(f"{gen}: {len(list(src.glob('*.npz')))} shards → mixed train pool")

# SDv1.4's own train embeddings (n_aug=2) — the head's home-field data
for shard in sorted(Path("/content/emb/train").glob("shard_*.npz")):
    dst = POOL_TRAIN / f"shard_sdv14_{shard.name}"
    shutil.copy(shard, dst)
print(f"sdv14: {len(list(Path('/content/emb/train').glob('*.npz')))} shards → mixed train pool")

# Val: keep SDv1.4's own val for training metrics; per-gen eval uses each gen's shards
for shard in sorted(Path("/content/emb/val").glob("shard_*.npz")):
    shutil.copy(shard, POOL_VAL / shard.name)

total = len(list(POOL_TRAIN.glob("shard_*.npz")))
print(f"\nmixed train pool: {total} shards")
print(f"mixed val pool: {len(list(POOL_VAL.glob('*.npz')))} shards (sdv14 val)")

Midjourney: 63 shards → mixed train pool
ADM: 63 shards → mixed train pool
BigGAN: 63 shards → mixed train pool
glide: 63 shards → mixed train pool
sdv14: 938 shards → mixed train pool

mixed train pool: 1190 shards
mixed val pool: 63 shards (sdv14 val)


In [21]:
from training.train_head import train
from training.calibrate_eval import calibrate

ckpt_mixed = train("/content/emb_mixed/train", "/content/emb_mixed/val",
                   out="/content/head_mixed.pt", epochs=5, device="cuda")
report = calibrate("/content/head_mixed.val_logits.npz",
                   "/content/head_mixed.calibration.json")
print(report)

train: 38000 samples (7.5 GB fp16), val: 2000 samples, dim=384, device=cuda
epoch 0: train_loss=0.3570 val_acc=0.8990
epoch 1: train_loss=0.2061 val_acc=0.9125
epoch 2: train_loss=0.1384 val_acc=0.9165
epoch 3: train_loss=0.0994 val_acc=0.9165
epoch 4: train_loss=0.0710 val_acc=0.9195
{'temperature': 1.821, 'ece_before': 0.0357, 'ece_after': 0.0203, 'nll_before': 0.2437, 'nll_after': 0.2028}


### Mixed vs. SDv1.4-only: the comparison

In [22]:
import json, numpy as np, torch
from training.train_head import build_head, _load_split, _batched_logits
from training.calibrate_eval import cross_generator_table, _sigmoid

GENS = ["Midjourney", "ADM", "BigGAN", "glide"]

def load_and_score(ckpt_path, cal_path):
    state = torch.load(ckpt_path, map_location="cpu")
    head = build_head(state["dim"]); head.load_state_dict(state["state_dict"])
    head = head.to("cuda").eval()
    T = json.load(open(cal_path))["temperature"]
    def score(emb_dir):
        p, c, y = _load_split(emb_dir)
        logits = _batched_logits(head, p, c, "cuda").numpy().astype(np.float64)
        return _sigmoid(logits / T), y.astype(np.float64)
    return score

print("═══ SDv1.4-only head ═══")
score_v1 = load_and_score("/content/head.pt", "/content/head.calibration.json")
r1 = {"sdv14 (in-dist)": score_v1("/content/emb/val")}
for gen in GENS: r1[gen] = score_v1(f"/content/eval/{gen}/emb")
print(cross_generator_table(r1))

print("\n═══ Mixed-generator head ═══")
score_mix = load_and_score("/content/head_mixed.pt", "/content/head_mixed.calibration.json")
r2 = {"sdv14 (in-dist)": score_mix("/content/emb/val")}
for gen in GENS: r2[gen] = score_mix(f"/content/eval/{gen}/emb")
print(cross_generator_table(r2))

═══ SDv1.4-only head ═══
| generator | AUROC | bal. acc |
|---|---|---|
| ADM | 0.345 | 0.453 |
| BigGAN | 0.560 | 0.524 |
| Midjourney | 0.832 | 0.700 |
| glide | 0.776 | 0.649 |
| sdv14 (in-dist) | 0.973 | 0.923 |
| **mean** | **0.697** | **0.650** |

═══ Mixed-generator head ═══
| generator | AUROC | bal. acc |
|---|---|---|
| ADM | 0.997 | 0.977 |
| BigGAN | 0.997 | 0.976 |
| Midjourney | 0.985 | 0.925 |
| glide | 0.991 | 0.947 |
| sdv14 (in-dist) | 0.975 | 0.919 |
| **mean** | **0.989** | **0.949** |


## Step 8 — Persist mixed-generator checkpoint

In [23]:
import json, shutil, subprocess
from pathlib import Path

commit = subprocess.run(["git", "-C", "/content/AI-image-Checkers", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
RUN = Path(f"/content/drive/MyDrive/ai_image_id/runs/{commit}_mixed")
RUN.mkdir(parents=True, exist_ok=True)

for f in ["/content/head_mixed.pt", "/content/head_mixed.calibration.json"]:
    shutil.copy(f, RUN)

(RUN/"provenance.json").write_text(json.dumps({
    "commit": commit, "backbone": "dinov2_vits14",
    "data": "GenImage mixed (sdv14+Midjourney+ADM+BigGAN+glide), val-split slices",
    "generators_in_training": ["sdv14", "Midjourney", "ADM", "BigGAN", "glide"],
    "n_per_class_per_gen": 1000, "sdv14_n_aug": 2, "others_n_aug": 0,
    "seeds": {"prep_sdv14": 0, "prep_others": 42},
}, indent=2))
print("saved to", RUN)

saved to /content/drive/MyDrive/ai_image_id/runs/8cf8026_mixed


## Roadmap
1. **Expand to 8 generators** — add VQDM, wukong, SDv1.5 to the mixed pool
   (same extract→embed→persist pattern).
2. **Real train/ run** — materialize 20K+/class from the train/ split (needs
   the streaming loader from `train_head_streaming.py` for RAM management).
3. **ONNX export** + pin the detector into CI with a tiny fixture.
4. **Interpretability** — score histograms, attention maps, perturbation
   sweeps (notebook 06).